# Commercial (shop) rent and site cost

**Coverage today:** cell-level shop rent exists **only for Dubai** (DLD Ejari shop contracts, `cell_shop_rent.csv`). Four Abu Dhabi mall lounges take a published JLL mall-class band (`retail_rent_bands.csv`). **Every other lounge and every growth area outside Dubai has no rent: other emirates are pending research.** Missing rent scores neutral on the cost signal and never blocks GROW.

**The model, as agreed on 2026-10-11** (one model for lounges and growth):
1. **Demand = served women** (`src/model/served.py`). Each cell's addressable women are split, Huff-style, across every premium salon and every lounge within 20 minutes of it, across emirate borders. Weight = reviews^ALPHA × (rating/4.6)^RATING_ELASTICITY × DECAY[drive-time band], plus an outside option (women who go elsewhere or use no premium salon). Drive-time bands come from the cell's own isochrones, else from straight-line bands fitted on those. Cells our salon search never covered get an imputed stand-in competitor (emirate × density tier).
2. **Cost = site cost** (`src/features/lounges.py`): shop rent × 150 m² + an assumed fit-out written off over 5 years. Cost efficiency = served women per AED 1k of site cost; `cost_index` = that ÷ the emirate's median lounge (1 = typical).
3. **Lounge scorecard** = served demand + cost index. Cannibalisation (the share of a lounge's women its siblings would absorb if it closed) is shown, not scored.
4. **Growth** = what-if: open a typical lounge (median lounge reviews and rating) at each area's centroid, and count the women it serves, how many it takes from our lounges, and the net new.

**Every parameter here is a first cut**, not calibrated. This notebook shows what they produce so we can decide together.

In [1]:
import sys, pathlib, statistics
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 120)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
from src.features import lounges as L
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
old = pd.read_csv(REPO_ROOT / "data/scenarios/baseline_v3_labels.csv")
old_action = dict(zip(old.id, old.action))
F = pd.DataFrame([x.model_dump() for x in r.features]).set_index("branch_id")
D = {d.branch_id: d for d in r.decisions}
AR = pd.DataFrame([x.model_dump(exclude={"cell_ids"}) for x in r.areas]).set_index("area_id")
AR["call"] = AR.index.map({d.area_id: d.action for d in r.area_decisions})

## 1. Dubai shop rent: distribution

In [2]:
shop = v3.cells.shop_rent_m2.dropna()
print("cells with a shop rent:", len(shop), "| by emirate:", v3.cells.loc[shop.index, "emirate"].value_counts().to_dict())
print(shop.describe(percentiles=[.1, .25, .5, .75, .9]).round(0))
area_rents = pd.read_csv(REPO_ROOT / "data/seed/v3/dubai_shop_rents_by_area.csv"); print("\nDLD areas file columns:", list(area_rents.columns))

cells with a shop rent: 243 | by emirate: {'Dubai': 243}
count     243.0
mean     1993.0
std       946.0
min        93.0
10%       889.0
25%      1364.0
50%      1985.0
75%      2441.0
90%      3095.0
max      5774.0
Name: shop_rent_m2, dtype: float64

DLD areas file columns: ['area', 'contracts', 'median_rent_m2', 'lat', 'lng', 'matched_to', 'match']


## 2. Does shop rent track the salon market?

Spearman, Dubai only. Salon level: each non-Bedashing salon against its cell's shop rent. Cell level: a cell's total salon reviews (all, and premium) against its shop rent. Also shop vs residential rent, cells with both.

In [3]:
from src.market import premium_substitutes, _reviews
sal = v3.salons[(v3.salons.excluded_reason == "") & ~v3.salons.is_bedashing.astype(bool)].reset_index()
c = v3.cells
lat, lng = sal.lat.to_numpy(float)[:, None], sal.lng.to_numpy(float)[:, None]
inbox = (c.south.to_numpy() <= lat) & (lat < c.north.to_numpy()) & (c.west.to_numpy() <= lng) & (lng < c.east.to_numpy())
sal["cell_id"] = np.where(inbox.any(1), c.index.to_numpy()[inbox.argmax(1)], None)
sal["reviews"] = [_reviews(x) for x in sal.to_dict("records")]
prem_ids = {x["place_id"] for x in premium_substitutes(sal.to_dict("records"), len(sal), a.premium_min_rating, a.comparable_price_levels)}
sal["premium"] = sal.place_id.isin(prem_ids)
sal["shop_rent"] = sal.cell_id.map(c.shop_rent_m2)
ds = sal[sal.shop_rent.notna()]
rows = [dict(level="salon", what="reviews, all salons", n=len(ds), spearman=ds.reviews.corr(ds.shop_rent, method="spearman")),
        dict(level="salon", what="reviews, premium salons", n=int(ds.premium.sum()),
             spearman=ds[ds.premium].reviews.corr(ds[ds.premium].shop_rent, method="spearman"))]
cell = pd.DataFrame({"shop": c.shop_rent_m2, "res": c.rent_observed,
                     "reviews": ds.groupby("cell_id").reviews.sum(), "prem_reviews": ds[ds.premium].groupby("cell_id").reviews.sum()})
cell = cell[cell.shop.notna()].fillna({"reviews": 0, "prem_reviews": 0})
rows += [dict(level="cell", what="total salon reviews", n=len(cell), spearman=cell.reviews.corr(cell.shop, method="spearman")),
         dict(level="cell", what="premium salon reviews", n=len(cell), spearman=cell.prem_reviews.corr(cell.shop, method="spearman")),
         dict(level="cell", what="residential rent (rent_observed)", n=int(cell.res.notna().sum()),
              spearman=cell.res.corr(cell.shop, method="spearman"))]
pd.DataFrame(rows).round(2)

,level,what,n,spearman
0,salon,"reviews, all salons",1177,0.08
1,salon,"reviews, premium salons",571,0.13
2,cell,total salon reviews,243,0.26
3,cell,premium salon reviews,243,0.23
4,cell,residential rent (rent_observed),217,0.33


## 3. Site cost: formula and the fit-out assumption

site cost (AED 1k/yr) = (shop rent/m² × REF_UNIT_M2 + FITOUT_AED_M2 × REF_UNIT_M2 ÷ FITOUT_YEARS) ÷ 1000

In [4]:
print("REF_UNIT_M2", L.REF_UNIT_M2, "|", L.REF_UNIT_WHY)
print("FITOUT_AED_M2", L.FITOUT_AED_M2, "| FITOUT_YEARS", L.FITOUT_YEARS, "|", L.FITOUT_WHY)
q = shop.quantile([.1, .25, .5, .75, .9])
pd.DataFrame({"shop_rent_m2": q, "site_cost_k": [L.site_cost_k(x) for x in q],
              "fitout_share_of_cost": [L.FITOUT_AED_M2 / L.FITOUT_YEARS / (x + L.FITOUT_AED_M2 / L.FITOUT_YEARS) for x in q]}).round(2)

REF_UNIT_M2 150 | Lounge floor areas aren't public: 150 m2 is an assumed reference unit. One size for every site, so it sets the level of women-per-rent, never the ranking.
FITOUT_AED_M2 3000 | FITOUT_YEARS 5 | Assumed, not sourced (2026-10-11): a salon fit-out of about AED 3,000/m² (AED 280/sqft), written off over 5 years, on the 150 m² reference unit: about AED 90k a year on top of rent.


,shop_rent_m2,site_cost_k,fitout_share_of_cost
0.10,889.2,223.38,0.40
0.25,1364.0,294.60,0.31
0.50,1985.0,387.75,0.23
0.75,2441.0,456.15,0.20
0.90,3095.0,554.25,0.16


## 4. Per lounge: served women, site cost, women per AED 1k, cost index

In [5]:
t = F[~F.not_scored][["emirate", "served_women", "site_rent_m2", "site_rent_source", "site_cost_k", "served_per_cost", "cost_index"]]
print("rent source:", t.site_rent_source.value_counts().to_dict())
t.sort_values(["emirate", "cost_index"]).round(2)

rent source: {'none': 14, 'dld': 5, 'band': 4}


,emirate,served_women,site_rent_m2,site_rent_source,site_cost_k,served_per_cost,cost_index
branch_id,,,,,,,
noya-plaza,Abu Dhabi,1385.72,1200.0,band,270.00,5.13,0.34
westyas,Abu Dhabi,3160.09,1200.0,band,270.00,11.70,0.77
shahama,Abu Dhabi,9865.31,2900.0,band,525.00,18.79,1.23
al-dhafra,Abu Dhabi,17633.50,1200.0,band,270.00,65.31,4.28
al-ain,Abu Dhabi,18651.84,NaN,none,NaN,NaN,NaN
al-falah,Abu Dhabi,9509.65,NaN,none,NaN,NaN,NaN
al-maqta,Abu Dhabi,6989.73,NaN,none,NaN,NaN,NaN
baniyas,Abu Dhabi,11086.02,NaN,none,NaN,NaN,NaN
delma,Abu Dhabi,1695.25,NaN,none,NaN,NaN,NaN


## 5. Emirate baseline (the median lounge) and the areas' cost index

In [6]:
base = t.dropna(subset=["served_per_cost"]).groupby("emirate").served_per_cost.agg(["count", "median"]).round(2)
print(base)
print("\nemirates with no lounge rent (cost_index None for every lounge and area):",
      sorted(set(t.emirate) - set(base.index)))

           count  median
emirate                 
Abu Dhabi      4   15.25
Dubai          5    5.36

emirates with no lounge rent (cost_index None for every lounge and area): ['Fujairah', 'Ras Al Khaimah', 'Sharjah']


In [7]:
ac = AR[AR.new_site_net.notna()]
print("areas with a what-if:", len(ac), "| with a cost index:", int(ac.cost_index.notna().sum()),
      "| by emirate:", ac[ac.cost_index.notna()].emirate.value_counts().to_dict())
ac[ac.cost_index.notna()][["emirate", "shop_rent_m2", "site_cost_k", "new_site_net", "served_per_cost", "cost_index", "call"]] \
    .sort_values("cost_index", ascending=False).round(2)

areas with a what-if: 160 | with a cost index: 69 | by emirate: {'Dubai': 69}


,emirate,shop_rent_m2,site_cost_k,new_site_net,served_per_cost,cost_index,call
area_id,,,,,,,
dubai-investments-park-dubai,Dubai,1096.0,254.40,4082.19,16.05,2.99,WATCH
dubai-investment-park-1-dubai,Dubai,1436.0,305.40,4282.61,14.02,2.61,WATCH
dubai-investment-park-2-dubai,Dubai,1096.0,254.40,3315.94,13.03,2.43,WATCH
warsan-dubai,Dubai,1011.0,241.65,3022.68,12.51,2.33,WATCH
al-khawaneej-2-dubai,Dubai,2722.0,498.30,6229.05,12.50,2.33,GROW
al-twar-dubai,Dubai,2099.0,404.85,4791.01,11.83,2.21,WATCH
al-rashidiya-dubai,Dubai,1696.0,344.40,3874.45,11.25,2.10,WATCH
umm-ramool-dubai,Dubai,1984.0,387.60,4324.16,11.16,2.08,WATCH
al-garhoud-dubai,Dubai,1237.0,275.55,3054.18,11.08,2.07,WATCH


## Open questions for review

1. **Rent coverage is thin.** Cell shop rent exists only for Dubai (243 cells). Four Abu Dhabi mall lounges use a JLL band. The other 14 of 23 lounges, and every area outside Dubai, have no rent: all 69 areas with a cost index are in Dubai. Sharjah, Ras Al Khaimah, Fujairah and Abu Dhabi street or villa sites are pending research.
2. **Abu Dhabi's baseline rests on four mall-band lounges** (median 15.25 women per AED 1k). al-dhafra's cost index of 4.28 comes from the community-mall band (AED 1,200/m²) applied to a remote catchment it dominates. A published band is not that site's rent.
3. **Shop rent barely tracks the salon market** (section 2): Spearman 0.08 for all salons and 0.13 for premium salons at salon level, 0.23–0.26 at cell level, and 0.33 against residential rent. Rent is a cost, not a demand proxy, which supports keeping it out of demand.
4. **The fit-out is unsourced,** and it is 16%–40% of site cost (rent p90 to p10). It narrows the gap between cheap and dear sites. Do we have a real fit-out figure?
5. **The area cost index compares net new women per AED 1k with what our lounges serve per AED 1k** (net vs gross), and DLD area medians mix mall and street shops (city-walk's site rent is AED 4,561/m²).
6. **al-barsha sits exactly at the Dubai median** (cost index 1.00), so its cost score of 0.5 is the same as "no rent".